In [1]:
!pip install nltk spacy scikit-learn pandas

In [2]:
import nltk
nltk.download("stopwords")

import spacy
!python -m spacy download en_core_web_sm

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 8.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
import pandas as pd

df = pd.read_csv("/content/resume_dataset_1200.csv")

print("Shape:", df.shape)
df.head()

Shape: (1200, 14)


,Name,Age,Gender,Education_Level,Field_of_Study,Degrees,Institute_Name,Graduation_Year,Experience_Years,Current_Job_Title,Previous_Job_Titles,Skills,Certifications,Target_Job_Description
0,Akash Pillai,30,Non-Binary,Master's,Cybersecurity,Master's in Cybersecurity,University of Pennsylvania,2025,0,NaN,NaN,"Node.js, JavaScript, Deep Learning, Statistics...",Google Cloud Professional,Seeking a challenging role as a Software Devel...
1,Charlotte Taylor,27,Non-Binary,Bachelor's,Electronics Engineering,Bachelor's in Electronics Engineering,Pune University,2019,5,Cybersecurity Engineer,NaN,"Spark, Kubernetes, Terraform, Natural Language...",TensorFlow Developer Certificate,Targeting a Cybersecurity Engineer position to...
2,James Zhou,45,Male,Bachelor's,Computer Science,Bachelor's in Computer Science,Amity University,2023,2,Prompt Engineer,NaN,"Data Analysis, Node.js, Machine Learning, Linu...","Microsoft Azure Fundamentals, Cisco Certified ...",Targeting a Prompt Engineer position to utiliz...
3,Amelia Thomas,28,Male,Master's,Information Technology,Master's in Information Technology,University of Pennsylvania,2023,2,AI Engineer,NaN,"Docker, Kubernetes, Blockchain, Spark",NaN,Targeting a Data Scientist position where I ca...
4,Amanda Jain,42,Non-Binary,Master's,Electronics Engineering,Master's in Electronics Engineering,University of Toronto,2023,2,Cybersecurity Engineer,NaN,"Statistics, Blockchain, Kubernetes, Cybersecurity",NaN,Looking for an opportunity as a Prompt Enginee...


In [4]:
df.columns

Index(['Name', 'Age', 'Gender', 'Education_Level', 'Field_of_Study', 'Degrees',
       'Institute_Name', 'Graduation_Year', 'Experience_Years',
       'Current_Job_Title', 'Previous_Job_Titles', 'Skills', 'Certifications',
       'Target_Job_Description'],
      dtype='object')

Rank resumes based on relevance to a job description

feature enginnering

In [6]:
def build_resume_text(row):
  return " ".join([
      str(row["Skills"]),
      str(row["Certifications"]),
      str(row["Current_Job_Title"]),
      str(row["Previous_Job_Titles"]),
      str(row["Field_of_Study"])
  ])
df["resume_text"] = df.apply(build_resume_text, axis=1)
df["resume_text"].head()

,resume_text
0,"Node.js, JavaScript, Deep Learning, Statistics..."
1,"Spark, Kubernetes, Terraform, Natural Language..."
2,"Data Analysis, Node.js, Machine Learning, Linu..."
3,"Docker, Kubernetes, Blockchain, Spark nan AI E..."
4,"Statistics, Blockchain, Kubernetes, Cybersecur..."


In [7]:
import re
from nltk.corpus import stopwords

In [8]:
STOPWORDS=set(stopwords.words("english"))
def clean_text(text):
  text=re.sub(r"[^a-zA-Z0-9]"," ",text.lower())
  tokens=[w for w in text.split() if w not in STOPWORDS]
  return " ".join(tokens)
df["clean_resume_text"]=df["resume_text"].apply(clean_text)


In [9]:
job_description=df["Target_Job_Description"].iloc[0]
clean_job_description=clean_text(job_description)
clean_job_description[:300]

'seeking challenging role software developer apply skills knowledge contribute organizational success professional growth'

 NLP VECTORIZATION (TF-IDF)
Vectorize Text

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer=TfidfVectorizer()
corpus=[clean_job_description]+df["clean_resume_text"].tolist()
tfidf_matrix = vectorizer.fit_transform(corpus)

SIMILARITY SCORING
 Cosine Similarity

In [11]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_scores = cosine_similarity(
    tfidf_matrix[0:1],
    tfidf_matrix[1:]
)[0]

df["nlp_score"] = similarity_scores


Final Scoring

In [12]:
df["experience_score"] = df["Experience_Years"] / df["Experience_Years"].max()

df["final_score"] = (
    0.7 * df["nlp_score"] +
    0.3 * df["experience_score"]
)

In [14]:
ranked_df = df.sort_values("final_score", ascending=False)

ranked_df[[
    "Name",
    "Current_Job_Title",
    "Experience_Years",
    "final_score"
]].head(10)


,Name,Current_Job_Title,Experience_Years,final_score
897,Ashley Johnson,Content Manager,10,0.390391
926,Jackson Smith,Operations Manager,10,0.343553
828,Samantha Zhou,Counselor,10,0.342214
666,Joshua Anderson,Mobile Applications Developer,10,0.329792
733,Karan Sharma,Market Research Analyst,8,0.316566
527,Aryan Sharma,Software Developer,9,0.314096
769,Andrew Verma,Mental Health Practitioner,9,0.314001
431,Liam Hernandez,Software Developer,9,0.313860
399,Mason Jackson,MLOps Specialist,9,0.311295
202,Priya White,Quantum Computing Specialist,8,0.308824


In [15]:
ranked_df.to_csv("ranked_candidates.csv", index=False)


In [16]:
df["final_score"].describe()


,final_score
count,1200.000000
mean,0.087203
std,0.080839
min,0.000000
25%,0.017760
50%,0.063653
75%,0.137841
max,0.390391


In [17]:
ranked_df.groupby("Field_of_Study")["final_score"].mean().sort_values(ascending=False)


,final_score
Field_of_Study,
Supply Chain Management,0.128667
Education,0.113412
Operations Management,0.113098
Machine Learning,0.103690
Computer Engineering,0.103505
Software Engineering,0.100802
Pharmacy,0.100163
Nursing,0.099984
Chemical Engineering,0.099263


In [18]:
ranked_df[["Skills", "Current_Job_Title", "final_score"]].head(5)

,Skills,Current_Job_Title,final_score
897,"Presentation Skills, Critical Thinking, Medica...",Content Manager,0.390391
926,"Project Management, Accounting, Medical Knowle...",Operations Manager,0.343553
828,"Customer Service, Critical Thinking, Project M...",Counselor,0.342214
666,"MongoDB, AWS, React, Azure, PyTorch, SQL",Mobile Applications Developer,0.329792
733,"Organizational Skills, Strategic Planning, Med...",Market Research Analyst,0.316566


In [19]:
!pip install sentence-transformers

In [20]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [21]:
resume_embeddings = model.encode(
    df["clean_resume_text"].tolist(),
    show_progress_bar=True
)

job_embedding = model.encode(
    clean_job_description
)


Batches:   0%|          | 0/38 [00:00<?, ?it/s]

In [22]:
from sklearn.metrics.pairwise import cosine_similarity

semantic_scores = cosine_similarity(
    [job_embedding],
    resume_embeddings
)[0]

df["bert_score"] = semantic_scores


In [23]:
df["final_score_bert"] = (
    0.6 * df["bert_score"] +
    0.4 * df["experience_score"]
)

In [24]:
df.sort_values(
    "final_score_bert",
    ascending=False
)[[
    "Name",
    "Current_Job_Title",
    "Experience_Years",
    "final_score_bert"
]].head(10)


,Name,Current_Job_Title,Experience_Years,final_score_bert
897,Ashley Johnson,Content Manager,10,0.679030
926,Jackson Smith,Operations Manager,10,0.648049
828,Samantha Zhou,Counselor,10,0.644788
733,Karan Sharma,Market Research Analyst,8,0.633999
1195,Mason Johnson,Counselor,9,0.629332
811,James Verma,Digital Marketing Specialist,9,0.613686
794,Emily Verma,Content Manager,8,0.607712
783,Harper Jackson,Market Research Analyst,10,0.605078
1081,Andrew Menon,Brand Manager,8,0.592597
483,Christopher Garcia,Machine Learning Engineer,9,0.592336


In [25]:
IT_KEYWORDS = [
    "engineer", "developer", "data", "ml",
    "ai", "cloud", "devops", "software"
]

def label_job(title):
    title = str(title).lower()
    return int(any(k in title for k in IT_KEYWORDS))

df["is_it"] = df["Current_Job_Title"].apply(label_job)


In [26]:
from sklearn.model_selection import train_test_split

X = resume_embeddings
y = df["is_it"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [28]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(random_state=42)
clf.fit(X_train, y_train)
print("Logistic Regression model initialized and trained.")

Logistic Regression model initialized and trained.


In [29]:
from sklearn.metrics import classification_report

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.87      0.85      0.86       172
           1       0.64      0.69      0.67        68

    accuracy                           0.80       240
   macro avg       0.76      0.77      0.76       240
weighted avg       0.81      0.80      0.81       240



## Summary:

### Data Analysis Key Findings

*   A Logistic Regression model was successfully trained on the provided training data (`X_train`, `y_train`).
*   The model achieved an overall accuracy of 80% on the test set (`X_test`, `y_test`).
*   For class 0, the model showed strong performance with a precision of 0.87, recall of 0.85, and an f1-score of 0.86 (172 samples).
*   For class 1, the performance was lower, with a precision of 0.64, recall of 0.69, and an f1-score of 0.67 (68 samples).
*   The macro average f1-score, which treats both classes equally, was 0.76. The weighted average f1-score, considering the number of samples in each class, was 0.81.
